# QFC pending experiments (V10, V6, V5, V11) on a Colab GPU

Runtime -> Change runtime type -> **GPU** before running. This notebook only wraps `scripts/run_gpu.sh`; the protocol (seeds, sizes, 30 random masks, k) is fixed in that script and in `docs/PROJECT_STATE.md`.

Rules: results from smoke runs are never results; do not change the protocol to get a different answer; use the pinned environment (the script installs `requirements-ci.txt` into a fresh venv).

In [ ]:
# ---- parameters: EDIT THESE ----
REPO_URL = "https://github.com/gourav-ai-engineer/quantum.git"
COMMIT = ""            # REQUIRED: full or short commit hash to run (results are named by it)
RESULTS_ROOT = "/content/drive/MyDrive/qfc_results"   # mounted Drive; survives disconnects
STEP19_EVAL_SIZE = 256  # >= 256, or -1 for the full validation split
ONLY = "step18,step14,step13,step19"   # order: V10, V6, V5, V11   # also "step20" = Round 2 (H1/H2/H6); run it on its own
assert COMMIT, "set COMMIT to the commit hash you want to run"

In [ ]:
# GPU + Drive. The script itself re-checks torch.cuda.is_available() after installing torch.
!nvidia-smi
from google.colab import drive
drive.mount("/content/drive")
!apt-get -qq install -y python3-venv > /dev/null

In [ ]:
# Clone at the commit, recreate the pinned env, run the pre-flight, then the experiments in order.
# Existing non-empty result dirs are never overwritten (use --tag NAME for a rerun on the same commit).
!rm -rf qfc_repo && git clone {REPO_URL} qfc_repo && cd qfc_repo && git checkout --detach {COMMIT}
# Colab: `python3 -m venv` fails at ensurepip; pre-create the env with virtualenv (run_gpu.sh reuses an existing .venv-gpu).
!pip install -q virtualenv && cd qfc_repo && python3 -m virtualenv .venv-gpu
!cd qfc_repo && bash scripts/run_gpu.sh --commit {COMMIT} --results-root {RESULTS_ROOT} --only {ONLY} --step19-eval-size {STEP19_EVAL_SIZE}

In [ ]:
# Ledger rows for docs/PROJECT_STATE.md: every number is read from the JSON, nothing is typed by hand.
import glob, os, subprocess
for exp_dir in sorted(glob.glob(os.path.join(RESULTS_ROOT, "*", COMMIT[:12] + "*"))):
    out = subprocess.run(["qfc_repo/.venv-gpu/bin/python", "qfc_repo/scripts/make_ledger_row.py", exp_dir], capture_output=True, text=True)
    print(exp_dir); print(out.stdout or out.stderr); print()

In [ ]:
# Round 2 (ONLY = "step20"): print the pre-registered verdicts that scripts/round2_analysis.py wrote.
import glob, json, os
for d in sorted(glob.glob(os.path.join(RESULTS_ROOT, "round2_h1_h2_h6", COMMIT[:12] + "*"))):
    print(d)
    print(json.dumps(json.load(open(os.path.join(d, "round2_verdicts.json")))["verdicts"], indent=2))


## Afterwards (on your machine)
1. Copy the small JSONs (`*.json`, `run_meta.json`, the log) from Drive into the repo as `results/<experiment>/<commit>/` on a new branch.
2. Paste the ledger rows into the Results ledger in `docs/PROJECT_STATE.md`, apply the decision rule/tree exactly as written, and open a PR. Do not merge without the owner's go-ahead.
3. If the runtime disconnects mid-run, re-run the parameter cell and use `ONLY` with the remaining steps plus a `--tag` if a partial directory exists.